# Batch Process Fault Bucketing & Extraction Pipeline

This notebook allows you to:
1. View all folders under `data/input/12-05-26-aarya/fault-bucketing/`
2. Select one or multiple folders to process (each folder contains `traces/raw_trace.json`)
3. Run the fault bucketing and extraction pipeline on the selected folders
4. Save results to `data/output/12-05-26-aarya/` retaining folder structure

## 1. Import Required Libraries

In [ ]:
import asyncio
import json
import sys
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd
from datetime import datetime

# Import the pipeline function
from run_bucketing_and_extraction_pipeline import run_pipeline

print("✓ Libraries imported successfully")

## 2. Define Configuration Parameters

In [ ]:
# Base directories
BASE_DIR = Path(r"C:\Users\meemankgupta\Music\Project\infosys\certifier")
INPUT_BASE_DIR  = BASE_DIR / "data" / "input"  / "21-05-ravi-top10-pii" / "1960bc89-361a-4fcb-af86-699113f09ec9-langfuse" / "e8650990-3905-4e56-b458-1ddb84d3972e" / "fault-bucketing"
OUTPUT_BASE_DIR = BASE_DIR / "data" / "output" / "26-05-ravi-top10-pii"

# Load configuration
try:
    from utils.load_config import ConfigLoader
    config = ConfigLoader.load_config()
    fault_config = config.get("fault_bucketing_config", {})
    pipeline_config = fault_config.get("pipeline", {})
    classifier_config = fault_config.get("classifier", {})
    BATCH_SIZE = pipeline_config.get("default_batch_size", 1)
    FAULT_PRUNING = classifier_config.get("fault_pruning", True)
    CACHE_ENABLED = classifier_config.get("cache_enabled", True)
    INCLUDE_EVENT_INPUT = classifier_config.get("include_event_input", False)
    PROMPT_PATH = classifier_config.get("prompt_path", None)
    STORE_TO_MONGODB = False
    print("✓ Configuration loaded successfully")
    print(f"\nPipeline Configuration:")
    print(f"  Batch Size: {BATCH_SIZE}")
    print(f"  Fault Pruning: {FAULT_PRUNING}")
    print(f"  Cache Enabled: {CACHE_ENABLED}")
    print(f"  Include Event Input: {INCLUDE_EVENT_INPUT}")
    print(f"  Store to MongoDB: {STORE_TO_MONGODB}")
except Exception as e:
    print(f"⚠ Could not load config: {e}")
    BATCH_SIZE = 1
    STORE_TO_MONGODB = False
    FAULT_PRUNING = True
    CACHE_ENABLED = True
    INCLUDE_EVENT_INPUT = False
    PROMPT_PATH = None

print(f"\nDirectories:")
print(f"  Input directory : {INPUT_BASE_DIR}")
print(f"  Output directory: {OUTPUT_BASE_DIR}")
print(f"  Input exists    : {INPUT_BASE_DIR.exists()}")

## 3. Define Helper Functions

Function to scan directories and find folders containing `traces/raw_trace.json` files.

In [ ]:
def find_trace_files(base_dir: Path) -> List[Dict[str, Any]]:
    """
    Find all raw_trace.json files. folder_id (run UUID) is already unique —
    no index prefix needed. Marks runs with existing metrics as 'done' so
    the processing loop can skip them.
    """
    trace_files = []
    for folder in sorted(base_dir.iterdir(), key=lambda p: p.name):
        if folder.is_dir():
            trace_path = folder / "traces" / "raw_trace.json"
            if trace_path.exists():
                folder_id = folder.name
                output_path = OUTPUT_BASE_DIR / folder_id
                # Check if this run was already extracted (has at least one metrics file)
                existing_metrics = list(output_path.rglob("*_metrics.json"))
                status = "done" if existing_metrics else "pending"
                trace_files.append({
                    "folder_id": folder_id,
                    "trace_path": str(trace_path),
                    "output_path": str(output_path),
                    "status": status,
                    "existing_metrics": len(existing_metrics),
                })
    return trace_files

print("✓ Utility functions defined")

## 4. Select Folders to Process

View available folders (each contains `traces/raw_trace.json`) and select which ones to process.

In [ ]:
# Discover all folders containing raw_trace.json
all_trace_files = find_trace_files(INPUT_BASE_DIR)
print(f"Found {len(all_trace_files)} folders with raw_trace.json files\n")

print("AVAILABLE FOLDERS:")
print("=" * 100)
for i, trace_info in enumerate(all_trace_files):
    print(f"  [{i:2d}]   {trace_info['folder_id']}")
print("=" * 100)

# Process ALL folders
selected_indices = list(range(0, len(all_trace_files)))
selected_trace_files = [all_trace_files[i] for i in selected_indices]

print(f"\n✓ SELECTED {len(selected_trace_files)} FOLDER(S) TO PROCESS:")
print("=" * 100)
for i, trace_info in enumerate(selected_trace_files):
    print(f"  [{selected_indices[i]}] {trace_info['folder_id']}")
    print(f"      → {trace_info['trace_path']}")
print("=" * 100)

## 5. Process Selected Folders Sequentially

Process each selected folder's `raw_trace.json` file **one at a time**. Each trace is fully processed and output saved before moving to the next.

In [ ]:
# results_summary = []
# start_time = datetime.now()

# print(f"Starting sequential processing of {len(selected_trace_files)} folder(s) at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
# print("=" * 80)

# for idx, trace_info in enumerate(selected_trace_files, 1):
#     folder_id   = trace_info["folder_id"]
#     trace_path  = trace_info["trace_path"]
#     output_path = trace_info["output_path"]   # OUTPUT_BASE_DIR / folder_id  (UUID, already unique)

#     print(f"\n[{idx}/{len(selected_trace_files)}] Folder: {folder_id}")
#     print(f"  → Input : {trace_path}")
#     print(f"  → Output: {output_path}")

#     # Skip runs that already completed to avoid deleting their fault_buckets
#     if trace_info.get("status") == "done":
#         print(f"  ⏭  SKIPPED — {trace_info['existing_metrics']} metrics file(s) already exist")
#         results_summary.append({
#             "folder_id": folder_id, "output_path": output_path,
#             "status": "skipped", "faults_extracted": trace_info["existing_metrics"],
#             "total_tokens": 0, "error": None,
#         })
#         continue

#     try:
#         results = await run_pipeline(
#             trace_file=trace_path,
#             output_dir=output_path,
#             batch_size=BATCH_SIZE,
#             store_to_mongodb=STORE_TO_MONGODB,
#             fault_pruning=FAULT_PRUNING,
#             cache_enabled=CACHE_ENABLED,
#             include_event_input=INCLUDE_EVENT_INPUT,
#             prompt_path=PROMPT_PATH,
#         )

#         total_tokens = sum(r["token_usage"]["total_tokens"] for r in results)
#         results_summary.append({
#             "folder_id": folder_id, "output_path": output_path,
#             "status": "success", "faults_extracted": len(results),
#             "total_tokens": total_tokens, "error": None,
#         })
#         print(f"  ✓ SUCCESS: {len(results)} faults extracted, {total_tokens:,} tokens used")

#     except Exception as exc:
#         results_summary.append({
#             "folder_id": folder_id, "output_path": output_path,
#             "status": "failed", "faults_extracted": 0,
#             "total_tokens": 0, "error": str(exc),
#         })
#         print(f"  ✗ FAILED: {exc}")

# duration = datetime.now() - start_time
# print("\n" + "=" * 80)
# print(f"Done in {duration}. Success: {sum(1 for r in results_summary if r['status'] == 'success')}, "
#       f"Skipped: {sum(1 for r in results_summary if r['status'] == 'skipped')}, "
#       f"Failed: {sum(1 for r in results_summary if r['status'] == 'failed')}")

## 6. Generate Summary Report

In [ ]:
# # Create summary DataFrame with explicit columns so it works even when empty
# SUMMARY_COLUMNS = ["index", "folder_id", "output_path", "status", "faults_extracted", "total_tokens", "error"]
# df_summary = pd.DataFrame(results_summary, columns=SUMMARY_COLUMNS) if results_summary else pd.DataFrame(columns=SUMMARY_COLUMNS)

# # Display summary statistics
# print("=" * 80)
# print("BATCH PROCESSING SUMMARY")
# print("=" * 80)
# print(f"\nTotal traces processed: {len(df_summary)}")
# print(f"Successful: {len(df_summary[df_summary['status'] == 'success'])}")
# print(f"Failed: {len(df_summary[df_summary['status'] == 'failed'])}")
# print(f"\nTotal faults extracted: {df_summary['faults_extracted'].sum()}")
# print(f"Total tokens used: {df_summary['total_tokens'].sum():,}")
# if len(df_summary) > 0:
#     print(f"Average tokens per trace: {df_summary['total_tokens'].mean():.0f}")

# # Display full summary table
# print("\n" + "=" * 80)
# print("DETAILED RESULTS")
# print("=" * 80)
# display(df_summary)

# # Save summary to CSV
# OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
# summary_file = OUTPUT_BASE_DIR / "batch_processing_summary.csv"
# df_summary.to_csv(summary_file, index=False)
# print(f"\n✓ Summary saved to: {summary_file}")

# # Display failures if any
# failures = df_summary[df_summary['status'] == 'failed']
# if len(failures) > 0:
#     print(f"\n⚠ {len(failures)} traces failed:")
#     for _, row in failures.iterrows():
#         print(f"  - [{row['index']}] {row['folder_id']}: {row['error']}")

## 7. Run Full Certification Pipeline (Aggregation → Hypothesis → Report)

In [ ]:
from run_full_certification_pipeline import run_pipeline as run_full_pipeline

# ── Configuration ────────────────────────────────────────────────────────────
CERT_AGENT_ID   = "1960bc89-361a-4fcb-af86-699113f09ec9"
CERT_AGENT_NAME = "demoinfra2_pii"
CERT_ADVANCED   = True

CERT_METRICS_DIR = OUTPUT_BASE_DIR
CERT_OUTPUT_DIR  = OUTPUT_BASE_DIR / "cert_output"

print(f"Metrics dir : {CERT_METRICS_DIR}")
print(f"Output dir  : {CERT_OUTPUT_DIR}")
print(f"Agent ID    : {CERT_AGENT_ID}")
print(f"Agent name  : {CERT_AGENT_NAME}")
print(f"Advanced    : {CERT_ADVANCED}")
print(f"Metrics dir exists: {CERT_METRICS_DIR.exists()}")

cert_result = await run_full_pipeline(
    metrics_dir=str(CERT_METRICS_DIR),
    output_dir=str(CERT_OUTPUT_DIR),
    agent_id=CERT_AGENT_ID,
    agent_name=CERT_AGENT_NAME,
    advanced_analysis=CERT_ADVANCED,
)

CERT_JSON_PATH = CERT_OUTPUT_DIR / f"certification_report_{CERT_AGENT_ID}.json"
print(f"\n✓ Certification pipeline complete.")
print(f"  Cert JSON: {CERT_JSON_PATH}")

## 8. Generate HTML/PDF Report via Cert Reporter

In [ ]:
import sys

# cert_reporter uses relative imports — add its root to sys.path
CERT_REPORTER_DIR = str(BASE_DIR / "cert_reporter")
if CERT_REPORTER_DIR not in sys.path:
    sys.path.insert(0, CERT_REPORTER_DIR)

from pipeline.graph import run_pipeline as run_reporter_pipeline

REPORTER_OUTPUT_DIR = CERT_OUTPUT_DIR / "report"

print(f"Cert JSON  : {CERT_JSON_PATH}")
print(f"Report dir : {REPORTER_OUTPUT_DIR}")
print(f"Cert JSON exists: {CERT_JSON_PATH.exists()}")

reporter_state = run_reporter_pipeline(
    input_path=str(CERT_JSON_PATH),
    output_dir=str(REPORTER_OUTPUT_DIR),
    formats=["html", "pdf"],
)

html_path = reporter_state.get("html_path", "")
pdf_path  = reporter_state.get("pdf_path", "")

print("\n✓ Cert reporter complete.")
if html_path:
    print(f"  HTML → {html_path}")
if pdf_path:
    print(f"  PDF  → {pdf_path}")
for err in reporter_state.get("errors", []):
    print(f"  ⚠ {err}")